# Exercise 02 : State for Two Agents

- Two nodes that run in the same step need a reducer on every key they both write
- A node's input schema is the only state that node receives
- A checkpointer keeps one thread, and the history lists every checkpoint

You will run a triage node and a parts node on ticket T-1001, fix the shared
write, hide a stock note from triage, and read the thread history. The Aurelia
records are made up.

Worked examples to revisit : module 03, demo01 · state-schemas, demo02 · reducers,
demo03 · checkpointers, demo04 · history-and-time-travel.

Run the next cell first. It loads the course key and builds `llm`, the course model. Nothing below it depends on where the notebook runs.

In [ ]:
# Course setup : run this cell first.
# On the classroom laptop the key comes from .env in the repository root.
# In Colab, add it under the key icon in the left sidebar as COURSE_API_KEY.
import os
import sys
import subprocess
import warnings
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
COURSE_FILES = "https://raw.githubusercontent.com/FeikoWielsma/Multi-Agents-LangGraph-Student/main/"
ROOT = None

if IN_COLAB:
    # Pinned to match uv.lock ; generated by build/stamp_setup.py
    PINS = ["langchain[mcp]==1.4.6", "langchain-core==1.6.12", "langchain-openai==1.7.1", "langgraph==1.2.17", "langgraph-checkpoint-sqlite==3.1.2", "deepagents[quickjs]==0.7.30", "langchain-quickjs==0.3.9", "langsmith==0.14.12", "openai==3.29.0", "python-dotenv==1.2.4", "langchain-huggingface==1.2.3", "sentence-transformers==6.1.0", "langchain-text-splitters==1.1.3", "rank-bm25==0.2.2", "agentevals==0.0.9", "httpx==0.28.1", "pytest==9.1.1", "pandas==3.0.6"]
    if not os.getenv("COURSE_SKIP_INSTALL"):
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PINS], check=True)
    try:
        from google.colab import userdata
        os.environ.setdefault("COURSE_API_KEY", userdata.get("COURSE_API_KEY"))
    except Exception:
        pass
else:
    from dotenv import load_dotenv
    for folder in [Path.cwd(), *Path.cwd().parents]:
        if (folder / "pyproject.toml").exists():
            ROOT = folder
            load_dotenv(folder / ".env")
            break

PROVIDER = os.getenv("COURSE_PROVIDER", "course")      # "course" or "ollama"
if PROVIDER == "ollama":
    MODEL = os.getenv("COURSE_MODEL", "qwen3.5:4b")
    BASE_URL = "http://localhost:11434/v1"
    API_KEY = "ollama"
    THINKING_OFF = {"reasoning_effort": "none"}
else:
    MODEL = os.getenv("COURSE_MODEL", "openai/gpt-6-luna")  # xiaomi/mimo-v2.6-pro is the step up
    BASE_URL = os.getenv("COURSE_BASE_URL", "https://openrouter.ai/api/v1")
    API_KEY = os.getenv("COURSE_API_KEY")
    # OpenRouter has no Luna endpoint that honours temperature or parallel_tool_calls ;
    # seed is honoured, and routing prefers OpenAI so repeated runs hit one provider
    THINKING_OFF = {"reasoning": {"effort": "none"}, "provider": {"order": ["openai"]}}
    if not API_KEY:
        raise RuntimeError("COURSE_API_KEY is missing : copy .env.example to .env and paste the key")

from langchain_core._api import LangChainBetaWarning
warnings.filterwarnings("ignore", category=LangChainBetaWarning)

from langchain_openai import ChatOpenAI
from langchain_core.rate_limiters import InMemoryRateLimiter

# One shared limiter paces every model call in this process ; COURSE_RPS sets the budget.
# Paid OpenRouter models have no platform request cap (measured 26 October 2026)
RATE_LIMITER = InMemoryRateLimiter(requests_per_second=float(os.getenv("COURSE_RPS", "2")),
                                   check_every_n_seconds=0.1, max_bucket_size=1)


def make_llm(**kw):
    """The course chat model. Keyword arguments pass straight to ChatOpenAI."""
    kw.setdefault("model", MODEL)
    if PROVIDER == "ollama":
        kw.setdefault("temperature", 0)
    else:
        kw.setdefault("seed", 7)
    kw.setdefault("extra_body", THINKING_OFF)
    kw.setdefault("timeout", 90)        # seconds per request ; a stuck call raises instead of hanging
    kw.setdefault("rate_limiter", RATE_LIMITER)
    return ChatOpenAI(base_url=BASE_URL, api_key=API_KEY, **kw)


def make_embeddings(**kw):
    """Local sentence embeddings, so retrieval costs no API calls."""
    from langchain_huggingface import HuggingFaceEmbeddings
    kw.setdefault("model_name", "sentence-transformers/all-MiniLM-L6-v2")
    return HuggingFaceEmbeddings(**kw)


def data(name):
    """Path to a course data file ; downloaded when there is no local copy."""
    if ROOT and (ROOT / "data" / name).exists():
        return str(ROOT / "data" / name)
    target = Path(name)
    if not target.exists():
        import urllib.request
        target.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(COURSE_FILES + "data/" + name, target)
    return str(target)


def data_folder(name):
    """Path to a course data folder ; downloaded file by file, using its index.txt, when there is no local copy."""
    if ROOT and (ROOT / "data" / name).is_dir():
        return str(ROOT / "data" / name)
    for file in Path(data(f"{name}/index.txt")).read_text(encoding="utf-8").split():
        data(f"{name}/{file}")
    return name


llm = make_llm()
print(f"model {MODEL} via {PROVIDER}")

## Step 1 : Shared key

Both nodes write `note` in the same step. Add the two edges from `START`.

In [ ]:
import csv
import sqlite3
from pathlib import Path

with open(data("capstone/tickets.csv"), newline="", encoding="utf-8") as handle:
    ticket = next(row for row in csv.DictReader(handle) if row["ticket_id"] == "T-1001")
print(ticket["ticket_id"], "|", ticket["symptom"], "|", ticket["parts_used"])

In [ ]:
from typing import Annotated, TypedDict

from langgraph.graph import END, START, StateGraph


class Clash(TypedDict):
    ticket_id: str
    note: str


def triage(state: Clash) -> dict:
    print("triage read :", sorted(state))
    return {"note": "triage " + state["ticket_id"]}


def parts(state: Clash) -> dict:
    print("parts read :", sorted(state))
    return {"note": "parts " + state["ticket_id"]}


clash = StateGraph(Clash)
clash.add_node("triage", triage)
clash.add_node("parts", parts)
clash.add_edge(____, "triage")
clash.add_edge(____, "parts")
clash.add_edge("triage", END)
clash.add_edge("parts", END)
clash_graph = clash.compile()
print("graph built")

<details><summary>Hint</summary>

`START` on both edges from the start.

</details>

In [ ]:
from langgraph.errors import InvalidUpdateError

try:
    clash_graph.invoke({"ticket_id": ticket["ticket_id"], "note": ""})
    print("check : fail, both writes were accepted")
except InvalidUpdateError as exc:
    print("check : pass", type(exc).__name__)
    print(str(exc).splitlines()[0])

## Step 2 : Reducer

Give `findings` a reducer that appends, then run both nodes again.

In [ ]:
import operator


class Shared(TypedDict):
    ticket_id: str
    symptom: str
    findings: ____


seen = {}


def triage(state: Shared) -> dict:
    seen["triage"] = sorted(state)
    print("triage read :", seen["triage"])
    return {"findings": [f"triage {state['ticket_id']} {state['symptom'][:48]}"]}


def parts(state: Shared) -> dict:
    seen["parts"] = sorted(state)
    print("parts read :", seen["parts"])
    return {"findings": [f"parts {state['ticket_id']}"]}


shared = StateGraph(Shared)
shared.add_node("triage", triage)
shared.add_node("parts", parts)
shared.add_edge(START, "triage")
shared.add_edge(START, "parts")
shared.add_edge("triage", END)
shared.add_edge("parts", END)
shared_graph = shared.compile()

<details><summary>Hint</summary>

`Annotated[list[str], operator.add]`.

</details>

In [ ]:
reduced = shared_graph.invoke({"ticket_id": ticket["ticket_id"], "symptom": ticket["symptom"], "findings": []})
print("findings :", reduced["findings"])
text = " ".join(reduced["findings"])
ok = "triage" in text and "parts" in text and seen["triage"] == seen["parts"]
print("check :", "pass" if ok else "fail")

## Step 3 : Private key

`stock_note` is the stock line for the ticket's part. The parts node reads it.
Triage does not. The invoke result leaves it out.

In [ ]:
class SharedInput(TypedDict):
    ticket_id: str
    symptom: str
    part_no: str


class TriageView(TypedDict):
    ticket_id: str
    symptom: str


class PartsView(TypedDict):
    ticket_id: str
    part_no: str
    stock_note: ____


class Public(TypedDict):
    ticket_id: str
    findings: list[str]


class Desk(TypedDict):
    ticket_id: str
    symptom: str
    part_no: str
    findings: Annotated[list[str], operator.add]
    stock_note: str


def lookup(state: SharedInput) -> dict:
    database = Path(data("capstone/service.db")).resolve()
    with sqlite3.connect(f"file:{database.as_posix()}?mode=ro", uri=True) as db:
        name, stock, depot = db.execute(
            "SELECT name, stock, depot FROM parts WHERE part_no = ?", (state["part_no"],)).fetchone()
    return {"stock_note": f"{state['part_no']} {name} stock {stock} at {depot}"}


def triage(state: TriageView) -> dict:
    seen["triage"] = sorted(state)
    print("triage read :", seen["triage"])
    return {"findings": [f"triage {state['ticket_id']} {state['symptom'][:48]}"]}


def parts(state: PartsView) -> dict:
    seen["parts"] = sorted(state)
    print("parts read :", seen["parts"])
    return {"findings": [f"parts {state['stock_note']}"]}


desk = StateGraph(Desk, input_schema=SharedInput, output_schema=____)
desk.add_node("lookup", lookup)
desk.add_node("triage", triage)
desk.add_node("parts", parts)
desk.add_edge(START, "lookup")
desk.add_edge("lookup", "triage")
desk.add_edge("lookup", "parts")
desk.add_edge("triage", END)
desk.add_edge("parts", END)
private_graph = desk.compile()
inputs = {"ticket_id": ticket["ticket_id"], "symptom": ticket["symptom"], "part_no": ticket["parts_used"]}

<details><summary>Hint</summary>

`str` on `stock_note`, and `Public` as `output_schema`.

</details>

In [ ]:
out = private_graph.invoke(inputs)
streamed = list(private_graph.stream(inputs, stream_mode="values"))
print("invoke keys :", sorted(out))
print("findings :", out["findings"])
for part in streamed:
    if "stock_note" in part:
        print("stream :", part["stock_note"])
hidden = "stock_note" not in out
visible = any("stock_note" in part and "stock 0" in part["stock_note"] for part in streamed)
ok = hidden and visible and "stock_note" not in seen["triage"] and "stock_note" in seen["parts"]
print("check :", "pass" if ok else "fail")

## Step 4 : History

Compile the same builder with a checkpointer and a thread, then read the history.

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

saved = desk.compile(checkpointer=____)
cfg = {"configurable": {"thread_id": ____}}
saved.invoke(inputs, cfg)
history = list(____)
print("checkpoints :", len(history))

<details><summary>Hint</summary>

`InMemorySaver()`, `"T-1001"`, and `saved.get_state_history(cfg)`.

</details>

In [ ]:
latest = saved.get_state(cfg)
print("thread findings :", latest.values["findings"])
for snap in history:
    print("step :", snap.metadata["step"], "next :", snap.next)
ok = len(history) >= 3 and len(latest.values["findings"]) == 2 and any(snap.next == () for snap in history)
print("check :", "pass" if ok else "fail")

## If time permits

- Open a second thread id on `saved` and compare `get_state` on both
- Replace `InMemorySaver` with `SqliteSaver` and open the same thread again
- Fork the checkpoint before `parts` with `update_state` and change `stock_note`